3Dslicer full airway analysis with Direct PHT for Dataset 1.

5 longest persistence for N directions for each Patient, resulting in np.array with shape 15*5N. The elements for each row should be ordered as follows.
$[p_{1,1},p_{1,2},...,p_{1,5},p_{2,1},p_{2,2},...,p_{2,5},........,p_{N,1},...,p_{N,5}]$, where $p_{i,j}$ is the jth largest persistence for direction i.

In [1]:
import json, pathlib, pandas as pd
import os, csv
from persistent_homology import (
    BettiZero,
    compute_intervals,
    compute_n_largest_bars,
    octahedron_directions,
)

In [2]:
def read_csv(path):
    with open(path, newline="") as f:
        reader = csv.reader(f)
        output = []
        for row in reader:
            values = []
            for value in row:
                if len(row) == 3:
                    values.append(float(value))
                elif len(row) == 2:
                    values.append(int(value))
            output.append(values)
        return output
    
def get_subfolders(path):
    """
    Return a list of names of all subdirectories in the given path.
    """
    return [
        name for name in os.listdir(path)
        if os.path.isdir(os.path.join(path, name))
    ]

def load_vertices_edges(seg_folder_path, seg_folder):
    folder_path = '{}/{}'.format(seg_folder_path, seg_folder)
    edges_path = '/edges.csv'
    vertices_path = '/vertices.csv'
    verts = read_csv(folder_path + vertices_path)
    edges = read_csv(folder_path + edges_path)
    return verts, edges

def process_direction(args):
    direction, vertices, edges = args
    bz = BettiZero(direction, vertices, edges)
    comps, mergers, verts, births = bz.compute_persistence()
    intervals = compute_intervals(births, mergers)
    bars = compute_n_largest_bars(intervals,5)
    return bars

In [3]:
json_data = {}

segmentation_folder_path = "./3Dslicer-full-lung-segmentations-processed"
lung_segmentations = get_subfolders(segmentation_folder_path)

directions = octahedron_directions()
for seg_folder in lung_segmentations:
    vertices, edges = load_vertices_edges(segmentation_folder_path, seg_folder)
    seg_data = []
    for index, direction in enumerate(directions):
        print("Direction: {}.".format(direction))
        largest_bars = process_direction((direction, vertices, edges))
        for bar in largest_bars:
            seg_data.append(bar[1]-bar[0]) 
        
    json_data[seg_folder] = seg_data
    print(f"✓ Processed {seg_folder}")
 
# Single JSON write at the end
with open("output_1.json", "w") as fp:
    json.dump(json_data, fp, indent=2)

Direction: [-1.  0.  0.].
Direction: [-0.99227788 -0.12403473  0.        ].
Direction: [-0.99227788  0.         -0.12403473].
Direction: [-0.99227788  0.          0.12403473].
Direction: [-0.99227788  0.12403473  0.        ].
Direction: [-0.96152395 -0.27472113  0.        ].
Direction: [-0.98019606 -0.14002801 -0.14002801].
Direction: [-0.98019606 -0.14002801  0.14002801].
Direction: [-0.96152395  0.         -0.27472113].
Direction: [-0.96152395  0.          0.27472113].
Direction: [-0.98019606  0.14002801 -0.14002801].
Direction: [-0.98019606  0.14002801  0.14002801].
Direction: [-0.96152395  0.27472113  0.        ].
Direction: [-0.89442719 -0.4472136   0.        ].
Direction: [-0.93704257 -0.31234752 -0.15617376].
Direction: [-0.93704257 -0.31234752  0.15617376].
Direction: [-0.93704257 -0.15617376 -0.31234752].
Direction: [-0.93704257 -0.15617376  0.31234752].
Direction: [-0.89442719  0.         -0.4472136 ].
Direction: [-0.89442719  0.          0.4472136 ].
Direction: [-0.93704257 